In [ ]:
import os
import numpy as np
import pandas as pd
import akshare as ak
from utils import get_stock_data

SEED = 42
START = '2023-04-01'
END = '2026-04-01'
CACHE_DIR = 'data/prices'
os.makedirs(CACHE_DIR, exist_ok=True)


def to_baostock_code(six_digit):
    """Convert a bare 6-digit A-share code to baostock format."""
    s = str(six_digit).zfill(6)
    prefix = s[0]
    if prefix in ('6', '9'):
        return f'sh.{s}'
    elif prefix in ('0', '2', '3'):
        return f'sz.{s}'
    elif prefix in ('4', '8'):
        return f'bj.{s}'
    raise ValueError(f'Unknown prefix for code: {s}')


def load_or_fetch(code, start, end, cache_dir=CACHE_DIR):
    """Load OHLCV from CSV cache if present, else fetch and cache."""
    cache_path = os.path.join(cache_dir, f'{code.replace(".", "_")}.csv')
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path, parse_dates=['date'], index_col='date')
        return df if len(df) > 0 else None
    try:
        df = get_stock_data(code, start, end)
        if df is None or len(df) == 0:
            return None
        df.to_csv(cache_path)
        return df
    except Exception as e:
        print(f'    fetch failed for {code}: {e}')
        return None


def sample_constituents(index_symbol, n=25, seed=42):
    """Sample n stocks from an index's current constituent list. Returns baostock-format codes."""
    cons_df = ak.index_stock_cons(symbol=index_symbol)
    code_col = None
    for candidate in ['品种代码', '成分券代码', '代码', 'code']:
        if candidate in cons_df.columns:
            code_col = candidate
            break
    if code_col is None:
        raise ValueError(f'Cannot find code column. Got: {cons_df.columns.tolist()}')
    codes = cons_df[code_col].astype(str).tolist()
    rng = np.random.default_rng(seed)
    sampled = rng.choice(codes, size=n, replace=False).tolist()
    return [to_baostock_code(c) for c in sampled]


def pull_basket(codes, label):
    """Pull or load OHLCV data for each code. Returns dict: code -> DataFrame."""
    data = {}
    print(f'Pulling {label} ({len(codes)} stocks)...')
    for i, code in enumerate(codes, start=1):
        df = load_or_fetch(code, START, END)
        rows = len(df) if df is not None else 0
        status = 'ok' if rows > 0 else 'FAILED'
        print(f'  [{i:>2}/{len(codes)}] {code}: {status} ({rows} rows)')
        if rows > 0:
            data[code] = df
    print(f'  {len(data)}/{len(codes)} succeeded\n')
    return data


def build_basket_returns(data, label, window_start):
    """Build wide-format returns, filter for full-history stocks, compute equal-weighted basket."""
    closes = pd.DataFrame({code: df['close'] for code, df in data.items()})
    first_valid = closes.apply(lambda s: s.first_valid_index())
    cutoff = pd.Timestamp(window_start) + pd.Timedelta('14 days')
    full_history = first_valid[first_valid <= cutoff].index.tolist()
    dropped = [c for c in closes.columns if c not in full_history]
    if dropped:
        print(f'{label}: dropping {len(dropped)} post-start listings: {dropped}')
    closes = closes[full_history]
    returns = closes.pct_change()
    basket = returns.mean(axis=1).dropna()
    print(f'{label}: n_stocks={len(full_history)}, n_days={len(basket)}, '
          f'first={basket.index[0].date()}, last={basket.index[-1].date()}')
    return basket, returns

In [ ]:
hs300_codes = sample_constituents('000300', n=25, seed=SEED)
zz1000_codes = sample_constituents('000852', n=25, seed=SEED)

print(f'HS300 sample ({len(hs300_codes)}): {hs300_codes}')
print()
print(f'ZZ1000 sample ({len(zz1000_codes)}): {zz1000_codes}')

pd.Series(hs300_codes, name='code').to_csv('data/hs300_sample_codes_25.csv', index=False)
pd.Series(zz1000_codes, name='code').to_csv('data/zz1000_sample_codes_25.csv', index=False)

In [ ]:
hs300_data = pull_basket(hs300_codes, 'HS300')
zz1000_data = pull_basket(zz1000_codes, 'ZZ1000')

In [ ]:
hs300_basket, hs300_returns = build_basket_returns(hs300_data, 'HS300', START)
zz1000_basket, zz1000_returns = build_basket_returns(zz1000_data, 'ZZ1000', START)

hs300_basket.to_csv('data/basket_returns_hs300_25stock.csv', header=['return'])
zz1000_basket.to_csv('data/basket_returns_zz1000_25stock.csv', header=['return'])
hs300_returns.to_csv('data/basket_component_returns_hs300_25stock.csv')
zz1000_returns.to_csv('data/basket_component_returns_zz1000_25stock.csv')

print('\n=== quick sanity check ===')
ann = np.sqrt(242)
print(f'HS300 basket:  ann_return={hs300_basket.mean()*242*100:6.2f}%   ann_std={hs300_basket.std()*ann*100:6.2f}%')
print(f'ZZ1000 basket: ann_return={zz1000_basket.mean()*242*100:6.2f}%   ann_std={zz1000_basket.std()*ann*100:6.2f}%')

In [ ]:
import numpy as np
import pandas as pd

TRADING_DAYS = 242
RF_ANNUAL = 0.018  # illustrative, 1y 国债 yield ~1.5-2%


def compute_drawdown(returns):
    """Return a DataFrame with cumulative equity, running max, and drawdown."""
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    dd = (cum - running_max) / running_max
    return pd.DataFrame({'cum': cum, 'running_max': running_max, 'drawdown': dd})


def risk_report(returns, label='', rf_annual=RF_ANNUAL, limits_series=None):
    """
    Compute a full risk report for a daily return series.

    Notes on conventions:
      - ann_return_arith: daily mean * 242, matches the standard used when
        scaling vol with sqrt(242). Use this when comparing to ann_std.
      - ann_return_geom: compounded actual path. Use this when describing
        the realised end-to-end growth.
      - Excess kurtosis (pandas default), so 0 = normal.
      - Sharpe and Sortino annualised with sqrt(242).
    """
    returns = returns.dropna()
    n = len(returns)
    ann_factor = np.sqrt(TRADING_DAYS)
    rf_daily = rf_annual / TRADING_DAYS
    excess = returns - rf_daily

    total_return = (1 + returns).prod() - 1
    ann_return_arith = returns.mean() * TRADING_DAYS
    ann_return_geom = (1 + total_return) ** (TRADING_DAYS / n) - 1
    ann_std = returns.std() * ann_factor

    sharpe = (excess.mean() / returns.std() * ann_factor) if returns.std() > 0 else np.nan
    downside = returns[returns < rf_daily]
    sortino = (excess.mean() / downside.std() * ann_factor) if (len(downside) > 1 and downside.std() > 0) else np.nan

    dd_df = compute_drawdown(returns)
    max_dd = dd_df['drawdown'].min()
    trough_date = dd_df['drawdown'].idxmin()
    pre_trough = dd_df.loc[:trough_date]
    peak_date = pre_trough['cum'].idxmax()
    dd_days_to_trough = (trough_date - peak_date).days
    recovery = dd_df.loc[trough_date:, 'drawdown']
    recovery_date = recovery[recovery >= 0].index[0] if (recovery >= 0).any() else None
    dd_recovery_days = (recovery_date - trough_date).days if recovery_date is not None else None

    report = {
        'label': label,
        'n_days': n,
        'total_return': total_return,
        'ann_return_arith': ann_return_arith,
        'ann_return_geom': ann_return_geom,
        'ann_std': ann_std,
        'sharpe': sharpe,
        'sortino': sortino,
        'max_drawdown': max_dd,
        'dd_peak_date': peak_date,
        'dd_trough_date': trough_date,
        'dd_days_to_trough': dd_days_to_trough,
        'dd_recovery_days': dd_recovery_days,
        'skewness': returns.skew(),
        'excess_kurtosis': returns.kurtosis(),
    }
    if limits_series is not None:
        aligned = limits_series.reindex(returns.index).fillna(False)
        n_hits = int(aligned.sum())
        report['limit_hit_count'] = n_hits
        report['limit_hit_fraction'] = n_hits / n
    return report


def print_risk_report(report):
    print(f"=== {report['label']} ===")
    print(f"  n_days:              {report['n_days']}")
    print(f"  total_return:        {report['total_return']*100:>7.2f}%")
    print(f"  ann_return (arith):  {report['ann_return_arith']*100:>7.2f}%")
    print(f"  ann_return (geom):   {report['ann_return_geom']*100:>7.2f}%")
    print(f"  ann_std:             {report['ann_std']*100:>7.2f}%")
    print(f"  sharpe (rf={RF_ANNUAL*100:.1f}%):    {report['sharpe']:>7.2f}")
    print(f"  sortino:             {report['sortino']:>7.2f}")
    print(f"  max_drawdown:        {report['max_drawdown']*100:>7.2f}%")
    print(f"  peak -> trough:      {report['dd_days_to_trough']:>3} days  "
          f"({report['dd_peak_date'].date()} -> {report['dd_trough_date'].date()})")
    rec = report['dd_recovery_days']
    print(f"  trough -> recovery:  {rec if rec is not None else 'not recovered'} days")
    print(f"  skewness:            {report['skewness']:>7.2f}")
    print(f"  excess kurtosis:     {report['excess_kurtosis']:>7.2f}")
    if 'limit_hit_count' in report:
        print(f"  limit hits:          {report['limit_hit_count']} "
              f"({report['limit_hit_fraction']*100:.2f}%)")
    print()


def _smoke_test_risk_report():
    """Hand-verified: constant positive returns → max_dd = 0, ann_std = 0, sharpe -> inf."""
    n = 250
    constant = pd.Series([0.001] * n, index=pd.date_range('2024-01-01', periods=n, freq='B'))
    r = risk_report(constant, 'smoke_constant')
    assert abs(r['max_drawdown']) < 1e-10, f"expected 0 drawdown, got {r['max_drawdown']}"
    assert abs(r['ann_std']) < 1e-10, f"expected 0 std, got {r['ann_std']}"
    assert r['n_days'] == n

    rng = np.random.default_rng(0)
    noise = pd.Series(rng.normal(0, 0.01, n), index=pd.date_range('2024-01-01', periods=n, freq='B'))
    r2 = risk_report(noise, 'smoke_noise')
    assert r2['max_drawdown'] < 0
    assert r2['ann_std'] > 0
    assert r2['dd_days_to_trough'] > 0

    print('risk_report smoke test passed.')


_smoke_test_risk_report()

In [ ]:
hs300_report = risk_report(hs300_basket, 'HS300 25-stock basket')
zz1000_report = risk_report(zz1000_basket, 'ZZ1000 22-stock basket')

print_risk_report(hs300_report)
print_risk_report(zz1000_report)

In [ ]:
import matplotlib.pyplot as plt
from plot_setup import setup_chinese_font
setup_chinese_font()

fig, (ax_cum, ax_dd) = plt.subplots(
    2, 1, figsize=(11, 8), sharex=True,
    gridspec_kw={'height_ratios': [2, 1]}
)

for basket, label, color in [
    (hs300_basket, 'HS300 (25)', '#185FA5'),
    (zz1000_basket, 'ZZ1000 (22)', '#D85A30'),
]:
    dd_df = compute_drawdown(basket)
    ax_cum.plot(dd_df.index, (dd_df['cum'] - 1) * 100,
                label=label, color=color, linewidth=1.5)
    ax_dd.fill_between(dd_df.index, dd_df['drawdown'] * 100, 0,
                       alpha=0.35, color=color, label=label)
    trough_date = dd_df['drawdown'].idxmin()
    trough_val = dd_df['drawdown'].min() * 100
    ax_dd.annotate(
        f'{label} trough\n{trough_date.strftime("%Y-%m-%d")}\n{trough_val:.1f}%',
        xy=(trough_date, trough_val), xytext=(15, -35),
        textcoords='offset points', fontsize=9, color=color,
        arrowprops=dict(arrowstyle='->', color=color, lw=0.8)
    )

ax_cum.set_ylabel('Cumulative return (%)')
ax_cum.set_title('HS300 25-stock vs ZZ1000 22-stock basket, 2023-04 to 2026-04')
ax_cum.axhline(0, color='gray', linewidth=0.5)
ax_cum.legend(loc='upper left')
ax_cum.grid(alpha=0.3)

ax_dd.set_ylabel('Drawdown (%)')
ax_dd.set_xlabel('Date')
ax_dd.axhline(0, color='gray', linewidth=0.5)
ax_dd.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data/basket_comparison_equity_drawdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
# In your notebook, append to utils.py:
# cat the risk_report, compute_drawdown, print_risk_report functions
# from Cell 5 into utils.py, then verify it imports cleanly:

import importlib
import utils
importlib.reload(utils)
from utils import risk_report, print_risk_report, compute_drawdown

# Re-run sanity check
test_report = risk_report(hs300_basket, 'HS300 25-stock (from utils)')
print_risk_report(test_report)


=== HS300 25-stock (from utils) ===
  n_days:              724
  total_return:          50.08%
  ann_return (arith):    15.10%
  ann_return (geom):     14.53%
  ann_std:               17.50%
  sharpe (rf=1.8%):       0.76
  sortino:                1.08
  max_drawdown:         -16.42%
  peak -> trough:      171 days  (2023-08-04 -> 2024-01-22)
  trough -> recovery:  119 days
  skewness:               0.26
  excess kurtosis:       12.56

